### If you get an error here, it means you need to run the prepare_data.ipynb file first

In [1]:
import json
import numpy as np
import pandas as pd

from pathlib import Path


description_path = Path('description.json')
if not description_path.exists():
    raise FileNotFoundError(
        "description.json not found. Please run 'prepare_data.ipynb' first to generate it."
    )

with open(description_path) as f:
    data = json.load(f)


## Preprocessing steps

1. **Load data** — Read the dataset and split columns into clinical variables and gene-level expression features.

2. **Low-expression gene filter (on author-provided processed data)** — Compute the 75th percentile of each gene across samples and keep genes with percentile value \(\ge 1\). This removes genes that are near-zero in most samples.

3. **Apply gene filter** — Subset the dataframe to clinical columns plus the retained genes.

4. **Report preprocessing effect** — Print the number of genes before and after filtering.

In [2]:
data_file_name = data['data_file_names'][0]

num_clinical_features = data['data_summary']['clinical_features']

df = pd.read_csv(data_file_name)

clinical_cols = df.columns[:num_clinical_features]
gene_cols = df.columns[num_clinical_features:]

print('Before preprocessing:\n - n_genes: %d' % (df[gene_cols].shape[1]))


robus_genes_mask = np.percentile(df[gene_cols], 75, axis=0) >= 1
# Apply mask to gene_cols and subset df to robustly-expressed genes only

# apply robust gene mask
gene_cols = df[gene_cols].columns[robus_genes_mask].tolist()
df = df[list(clinical_cols) + gene_cols]


print("\nAfter preprocessing:\n - n_genes: %d" % (df[gene_cols].shape[1]))

transformed_data_file_name = data_file_name.replace("original", "preprocessed")

Before preprocessing:
 - n_genes: 24748

After preprocessing:
 - n_genes: 12474


In [3]:
url = "https://www.ncbi.nlm.nih.gov/geo/download/?format=file&type=rnaseq_counts&file=Human.GRCh38.p13.annot.tsv.gz"
df_gene_annotations = pd.read_csv(
    url,
    sep="\t",
    compression="gzip"
)
df_gene_annotations = df_gene_annotations[df_gene_annotations["Status"] == "active"]
df_gene_annotations = df_gene_annotations[df_gene_annotations["GeneType"] == "protein-coding"]
df_gene_annotations = df_gene_annotations.reset_index(drop=True)

protein_coding_genes = set(df_gene_annotations["Symbol"].dropna())
gene_cols = [gene for gene in gene_cols if gene in protein_coding_genes]
df = df[list(clinical_cols) + gene_cols]

print("\nAfter protein-coding filter:\n - n_genes: %d" % len(gene_cols))

df.to_csv(transformed_data_file_name, index=False)

if transformed_data_file_name not in data['data_file_names']:
    data['data_file_names'].append(transformed_data_file_name)

with open(description_path, 'w') as f:
    json.dump(data, f, indent=4)

/var/folders/h1/fnfx16m50jvd8ddpcs93g9mw0000gn/T/ipykernel_16727/4029965179.py:2: DtypeWarning: Columns (8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  df_gene_annotations = pd.read_csv(



After protein-coding filter:
 - n_genes: 11715


## Survival variables encodings

Authors encode survival variables in various ways, here we have just a check if all the survival variables are already numeric. If they are not, we convert them by hand.


In [4]:
df_ = df.copy()

endpoints = []
for endpoint in data["survival-endpoints"]:
    endpoint_time = endpoint["time_var"]["var_name"]
    endpoint_event = endpoint["event_var"]["var_name"]

    if endpoint_time == "unknown" or endpoint_event == "unknown":
        continue

    # Replace "Not available" and similar missing-value strings with NaN
    for col in [endpoint_time, endpoint_event]:
        df_[col] = df_[col].replace(["Not available", "NA", "N/A", "not available", "-"], pd.NA)


    # make sure time and event are numeric
    df_[endpoint_time] = pd.to_numeric(df_[endpoint_time])
    df_[endpoint_event] = pd.to_numeric(df_[endpoint_event])
    
    endpoints.append((endpoint_time, endpoint_event))


# Flatten endpoints and use it to order survival columns
surv_cols = [col for tup in endpoints for col in tup]
other_cols = [col for col in df_.columns if col not in surv_cols]
ordered_cols = surv_cols + other_cols

df_[ordered_cols].to_csv(transformed_data_file_name, index=False)